# B 실험: Can 예산 135개 (Kaggle)

- 1주차 성공 기준 "학습 후보의 50% 이하 시범으로 전체 대비 하락 5%p 이내"를 문장 그대로 확인한다. 본 실험(150개)은 56%였다
- 학습 계획은 로컬 `select_demos(budget=135)` → `review` → `train_evaluate(prepare)`가 만든 `plan_b64.txt` (본 실험과 같은 분석·설정, 예산만 다름)
- 실행: 선별 135 · 무작위 135 x 시드 1·2·3. 전체 데이터 기준선은 본 실험 train 모델과 같아 다시 학습하지 않는다
- 평가: 200 epoch 30회 + 400 epoch 100회. 물체 위치와 행동 샘플링 난수를 에피소드마다 고정하고 위치별 성공/실패를 기록한다 (Can 짝 평가와 같은 위치·같은 난수)
- 판정 (결과 보기 전에 정함): 짝지은 비교, 시드 3개 x 시험 100개, 세 시드 모두 같은 방향일 때만 "차이 있음"
- 결과와 체크포인트는 `/kaggle/working/budget/` (세션 설정 Persistence = Files only). 꺼지면 다시 Run All → 끝난 실행은 건너뛴다

In [ ]:
# 1. GPU와 파이썬 버전 확인
import sys, torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('[결과] python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. 설치 (코랩 노트북과 같은 고정 버전, 작업 폴더만 /kaggle/temp)
#  - 저장소를 /content 바로 아래에 받으면 코랩이 sys.path에 /content를 넣어 두어서, 설치된 패키지 대신 저장소 폴더를 import 한다 → /content/src 아래에 받는다
#  - robosuite 1.5.1은 mujoco>=3.2.3 만 요구해 최신 3.15가 깔리는데, 관절 처리에서 AssertionError가 난다 → mujoco 3.3.0으로 고정
#  - robomimic은 transformers==4.41.2 등을 고정 요구해 Python 3.13에서 빌드 실패 → 의존성 없이 설치하고 필요한 패키지만 따로 설치
#  - 편집 가능 설치(-e)는 세션 재시작 전까지 import가 안 되므로 일반 설치를 쓴다
import time; t0 = time.time()
WORK = '/kaggle/temp'
%cd {WORK}
!mkdir -p src
![ -d src/robosuite ] || git clone -q https://github.com/ARISE-Initiative/robosuite.git src/robosuite
!cd src/robosuite && git checkout -q v1.5.1
![ -d src/robomimic ] || git clone -q https://github.com/ARISE-Initiative/robomimic.git src/robomimic
!pip install -q ./src/robosuite
!pip install -q --no-deps ./src/robomimic
!pip install -q psutil tqdm termcolor tensorboard tensorboardX imageio imageio-ffmpeg matplotlib egl_probe huggingface_hub transformers diffusers
# robosuite 설치 과정(mink 의존성)에서 mujoco가 최신으로 올라가므로, 마지막에 3.3.0으로 강제로 되돌린다
!pip install -q --no-deps --force-reinstall "mujoco==3.3.0"
import robosuite, robomimic, mujoco
print('[결과] robosuite', robosuite.__version__, '| mujoco', mujoco.__version__, '| robomimic', robomimic.__file__)
print(f'[결과] 설치 {time.time()-t0:.0f}초')

In [ ]:
# 3. 학습 계획 + 실행 목록
MODE = 'main'
PLAN_B64 = 'eNqtVttu20gM/Rc9s43IuWjkz+hrEAiKLdcGHDuwnSy2Rf59D0eUPXLj7S7QB8EWOZo55OEh52e1fq0WlV+tn5+XtazWwiH1Q0XV8W3fbVfwffsitUSuWb6cht2wPH95ZhewYtWf+9NwxpJlv+9eNjBt1t1rf97A9M5fwwPsDy+bh93hr261feneOXzdrNb67fKwX2+/V4ufVb/7fsD652V33O/h2b+9dMPrYbk5VQtf11SdzsPrqXsdjqO5WrBan/vzctOdtj8GM5yGAWgZwA+73eHt3O3NMb1vDsftj8Pedh364+7vbnjvd9O2cmPGSld/UHU4roZjtXislm/H/jysujH6Y79fHV7s5XSGa7vewjuz28v62L8MJ7xj2XZfPSna80mjL/dcPNbEJOQoUKKWGK9M7Ig9cSROxC1JTYJFjsSTBJJI0pAkkpZcTY7JYQNHzpML5CK5hlwi15KvyTN5Ie/Ie/KBfCTfkE/k9SwcVuO0WvDgyBqn1bBnELArDMXBAQ/AKBoAYcF6QGFgYYBhoGFRH74FIHYaAdYAEwMU4sSDdcDFAMZAxoDGwMZeQ8UaYGMgY0DjAFvAe9D/WBuxNsIWNSewR+wVYW9gb3BOA3sDW4PvG3zTwJfgS9g/wZ/wTdJkwp/gT/C38Lfwt/C32LvFHi3WtVjTataRdsQviFk07ZpxxCaISxCPIA5BHII4BHEIEisBv1Ef+HCmJGUJa3CWtP5pVj9X4lPmOhRce+NXyRWlVRlMFISQDSQDZyAVyAIOQg4aR8gAAka8CDdhS7yiniIBAQAgRoTYRkJwiK2dqFfaQ0F7MIpro/iG3l+onWi9pTMalbCH+g6dPKf0QmUYKcy0wY9IMmWZrokqpSkaVZoxpLpWulQmyCnSKZpJTaWKZqIP5TmnMI40ZgpVXPgPvBKC0YjvkORMJ8pNgFEa2BvVnxKEd6RcgPFKdzLKYQdeAV5pw9NtV1g8Xriv7wg7S9oEXKgXyQQ+qwOXuW8IBa/ch4JsP2PaxF2K+sJuKkTaGotxFGJmkEf2MmuTAFVksOP0zNhMbMaaCUo7rKDJZJYyO/obTVgTO6FgBjbXmLjYmIkFO8pGbWyEa+azyDTr7dO93rx4hIKm/hqz2sa8X9TmcvtMOe+XrhmtX+J0JMMISDMtonSBRSU46i+o8oAIgIyMcCs+/qTPppve2tyILlx7aBacs74Zr73zIjox4aVCcN4EFoseOYmsKXpivPZAlM4osFQIzJnIYiE0uVKpdZzpZKPTm9iaQnBKrRi98COOkWasQQy5lwJ/pnoSYeNNfCoyZ8KbBDeJTUshWRmMo/dfZ2z4ZczKn5+0VjnyWfG43MuDtXP0Ge3onCsq93UQhphV5Cnr/F6bT1n8Y8lJUXXxk8K7DPzw34Y+kjNOhWn4uz94AZCbS8BnxcxFQf9mkuTLQSouCGyTxdlFIfzmsjAJ4v9cGuL84nDpeTIX0GxKTSIq+uFFTGyCcnNR5SmWiknGRc/0xVRrPrmg3BNbMsHVJjoZhadTUHst8jyKUCeis6kY7kxGfIs8j0IN10mpgkWerxNTCgFr/44307O+maBepyieOBP4x8c/6GK4tQ=='
import json, base64, zlib, os
PLAN = json.loads(zlib.decompress(base64.b64decode(PLAN_B64)))
SETS = {k: [f'demo_{i}' for i in v] for k, v in PLAN['sets'].items()}
OUTDIR = '/kaggle/working/budget'
CKPT = f'{OUTDIR}/ckpt_{PLAN["fp"]}'
os.makedirs(CKPT, exist_ok=True)
CORE = ['curated_135', 'random_135']
JOBS = [(n, s) for s in (1, 2, 3) for n in CORE]
EVAL_N = {PLAN['config']['early_eval_epoch']: PLAN['config']['early_eval_n'], PLAN['config']['num_epochs']: PLAN['config']['rollout_n']}
print('[결과] 데이터', PLAN['dataset'], '| 지문', PLAN['fp'], '| 실행', len(JOBS), JOBS)
print('[결과] 설정', PLAN['config'], '| 평가', EVAL_N)

In [ ]:
# 4. 데이터 다운로드 (계획에 적힌 데이터셋)
os.environ['MUJOCO_GL'] = 'egl'   # 화면 없는 서버용 렌더링 설정
os.makedirs(f'{WORK}/data', exist_ok=True)
DATA = f"{WORK}/data/{PLAN['dataset']}.hdf5"
if not os.path.exists(DATA):
    !wget -q -O {DATA} 'https://huggingface.co/datasets/robomimic/robomimic_datasets/resolve/main/{PLAN["hf_path"]}?download=true'
!ls -la {DATA}

In [ ]:
# 5. 묶음별 필터 키를 넣은 학습용 데이터 복사본 (원본은 그대로)
import shutil, h5py, numpy as np
PDATA = f'{WORK}/data/plan.hdf5'
shutil.copy(DATA, PDATA)
with h5py.File(PDATA, 'a') as f:
    for k, ids in SETS.items():
        key = f'mask/{k}'
        if key in f:
            del f[key]
        f.create_dataset(key, data=np.array(ids, dtype='S'))
    print('[결과] 필터 키', {k: len(f[f'mask/{k}']) for k in SETS})

In [ ]:
# 6. 학습·평가 함수 (평가 난수 고정, 위치별 결과 기록, 체크포인트)
import contextlib, glob, math, shutil, time, torch, numpy as np
from robomimic.config import config_factory
from robomimic.scripts.train import train
import robomimic.utils.file_utils as FileUtils
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_SEED = 12345

def make_config(cfg, set_name, out_dir, save_every):
    config = config_factory(algo_name='bc')
    with config.values_unlocked():
        config.experiment.name = f"{set_name}_{cfg['algo']}"
        config.train.data = [{'path': PDATA}]
        config.train.output_dir = out_dir
        config.train.hdf5_filter_key = set_name
        config.train.num_epochs = cfg['num_epochs']
        config.experiment.epoch_every_n_steps = cfg['steps_per_epoch']
        config.train.batch_size = cfg['batch_size']
        config.train.seed = cfg['seed']
        config.train.cuda = torch.cuda.is_available()
        config.experiment.validate = False
        config.experiment.rollout.enabled = False
        config.experiment.render_video = False
        config.experiment.logging.log_tb = False
        config.experiment.save.enabled = True
        config.experiment.save.every_n_epochs = save_every
        config.experiment.save.on_best_validation = False
        config.experiment.save.on_best_rollout_return = False
        config.experiment.save.on_best_rollout_success_rate = False
        config.observation.modalities.obs.low_dim = ['robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos', 'object']
        if cfg['algo'] == 'bc_rnn':   # robomimic 논문의 low_dim BC-RNN 설정
            config.algo.rnn.enabled = True
            config.algo.rnn.horizon = 10
            config.algo.rnn.hidden_dim = 400
            config.algo.rnn.num_layers = 2
            config.train.seq_length = 10
            config.algo.gmm.enabled = True
            config.algo.actor_layer_dims = ()
    return config

def evaluate(ckpt, n, horizon):
    # i번째 에피소드: 물체 초기 위치(np)와 행동 샘플링(torch) 난수를 모두 EVAL_SEED + i로 고정
    policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=ckpt, device=DEV, verbose=False)
    env, _ = FileUtils.env_from_checkpoint(ckpt_dict=ckpt_dict, render=False, render_offscreen=False, verbose=False)
    out = []
    for i in range(n):
        np.random.seed(EVAL_SEED + i)
        torch.manual_seed(EVAL_SEED + i)
        policy.start_episode()
        obs = env.reset()
        ok = 0
        for t in range(horizon):
            obs, r, done, _ = env.step(policy(ob=obs))
            if env.is_success()['task']:
                ok = 1
                break
        out.append(ok)
    return out

def find_ckpt(set_name, seed, ep):
    hit = sorted(glob.glob(f'{CKPT}/{set_name}_s{seed}/model_epoch_{ep}.pth'))
    return hit[-1] if hit else None

def run_set(set_name, cfg, eval_n):
    eval_epochs = sorted(eval_n)
    seed = cfg['seed']
    dst = f'{CKPT}/{set_name}_s{seed}'
    ckpts = {ep: find_ckpt(set_name, seed, ep) for ep in eval_epochs}
    if all(ckpts.values()):
        train_sec = json.load(open(f'{dst}/train.json'))['train_sec'] if os.path.exists(f'{dst}/train.json') else 0
        print('[체크포인트] 저장된 모델로 평가만', set_name, seed, flush=True)
    else:
        out_dir = f"{WORK}/runs/{set_name}_s{seed}"
        shutil.rmtree(out_dir, ignore_errors=True)
        save_every = math.gcd(*eval_epochs)   # 모든 평가 epoch에 모델이 저장되게
        t0 = time.time()
        os.makedirs(f'{WORK}/logs', exist_ok=True)
        with open(f"{WORK}/logs/{set_name}_s{seed}.log", 'w') as lf, contextlib.redirect_stdout(lf), contextlib.redirect_stderr(lf):
            train(make_config(cfg, set_name, out_dir, save_every), device=DEV)
        train_sec = round(time.time() - t0)
        os.makedirs(dst, exist_ok=True)
        for ep in eval_epochs:   # 평가 전에 모델부터 저장 (평가 도중 꺼져도 다시 학습하지 않게)
            shutil.copy(sorted(glob.glob(f'{out_dir}/**/models/model_epoch_{ep}.pth', recursive=True))[-1], f'{dst}/model_epoch_{ep}.pth')
            ckpts[ep] = f'{dst}/model_epoch_{ep}.pth'
        json.dump({'train_sec': train_sec}, open(f'{dst}/train.json', 'w'))
    rows = {}
    for ep in eval_epochs:
        t1 = time.time()
        o = evaluate(ckpts[ep], eval_n[ep], cfg['rollout_horizon'])
        rows[str(ep)] = {'success': sum(o), 'rollouts': len(o), 'outcomes': o, 'eval_sec': round(time.time() - t1)}
    return train_sec, rows
print('[결과] 함수 준비, 장치', DEV)

In [ ]:
# 7. 실행: (묶음, 시드) 하나 끝날 때마다 결과 파일에 저장, 끝난 것은 건너뜀
import datetime
KST = datetime.timezone(datetime.timedelta(hours=9))
OUT = f"{OUTDIR}/results_{MODE}_{PLAN['fp']}.json"
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
doc = json.load(open(OUT)) if os.path.exists(OUT) else {
    'fingerprint': PLAN['fp'], 'run_id': PLAN['run_id'], 'dataset': PLAN['dataset'], 'mode': MODE, 'platform': 'kaggle',
    'gpu': gpu, 'eval_seed': EVAL_SEED, 'policy_seed': 'EVAL_SEED+i', 'time_zone': 'KST', 'config': PLAN['config'],
    'eval_n': {str(k): v for k, v in EVAL_N.items()}, 'runs': {}}
for name, seed in JOBS:
    key = f'{name}|seed{seed}'
    if key in doc['runs']:
        continue
    cfg = dict(PLAN['config'], seed=seed)
    train_sec, rows = run_set(name, cfg, EVAL_N)
    last = rows[str(max(EVAL_N))]
    doc['runs'][key] = {'set': name, 'seed': seed, 'success': last['success'], 'rollouts': last['rollouts'],
                        'train_sec': train_sec, 'eval_sec': sum(r['eval_sec'] for r in rows.values()), 'by_epoch': rows,
                        'finished_at': datetime.datetime.now(KST).isoformat(timespec='seconds')}
    with open(OUT + '.tmp', 'w') as f:
        json.dump(doc, f)
    os.replace(OUT + '.tmp', OUT)
    print('RESULT', key, {ep: f"{r['success']}/{r['rollouts']}" for ep, r in rows.items()}, 'train', train_sec, flush=True)
print('[결과] 완료', len(doc['runs']), '개 →', OUT)